Code that calculates the operator pool A, using the nested commutator approach to calculating the adiabatic gauge potential (arXiv:1904.03209), up to a given order and keeping up to n-body terms, for a given problem Hamiltonian and mixer Hamiltonian (it assumes that H_0 is an adiabatic Hamiltonian of the form H_ad = (1 - λ)H_M + λH_P). It returns A in the form of a list of Pauli operator terms. A term then is chosen from this list and summed over the qubits to get A_DC.

It calculates A numerically so there is a possibility of unintended cancellations for different problem and mixer Hamiltonians. You can check this by changing the value of the variable lambda_fn and seeing if you get the same result (but do not change it to 1 as that will cancel the mixer Hamiltonian in the calculation).

The code with the current variables runs instantly.

In [ ]:
# Imports the modules

import numpy as np
import itertools
from qiskit.quantum_info import SparsePauliOp

In [ ]:

# Defines the Schwinger model Hamiltonian for a given J, w, m, theta.
# Uses SparsePauliOp from qiskit. First generates a list of Pauli operators and then uses .from_sparse_list to generate the sparse Pauli operator.
# Outputs the Hamiltonian.

def Schwinger_Hamiltonian(num_qubits, J, w, m, theta):

    # Constructs the H_ZZ term.
    Paulis_ZZ = []
    for n in range(1, num_qubits):
        for k in range(1,n):
            Paulis_ZZ.append(("ZZ", [k-1, n-1], J*(num_qubits-n)/2))
    
    H_ZZ = SparsePauliOp.from_sparse_list(Paulis_ZZ, num_qubits)

    # Constructs the H_+- term.
    Paulis_pm = []
    for n in range(1,num_qubits):
        W_n = w-(-1)**n*m*np.sin(theta)/2
        Paulis_pm.append(("XX", [n-1, n], W_n/2))
        Paulis_pm.append(("YY", [n-1, n], W_n/2))
    
    H_pm = SparsePauliOp.from_sparse_list(Paulis_pm, num_qubits)

    # Constructs the H_Z term.
    Paulis_Z = []
    for n in range(1, num_qubits):
        C_n = np.floor(num_qubits/2) - np.floor(n/2)
        Paulis_Z.append(("Z", [n-1], (-1)**n*m*np.cos(theta)/2-J*C_n/2))
    Paulis_Z.append(("Z", [num_qubits-1], (-1)**num_qubits*m*np.cos(theta)/2))
    
    H_Z = SparsePauliOp.from_sparse_list(Paulis_Z, num_qubits)

    # Adds the terms together to get the full Hamiltonian.
    H_prob_Schwinger = H_ZZ + H_pm + H_Z
    # Removes terms from the Hamiltonian with very small coefficients which are meant to be zero as this can lead to errors in the optimizer.
    H_prob_Schwinger = H_prob_Schwinger.simplify(atol=1e-10)
    
    return H_prob_Schwinger

In [ ]:

# Constructs and returns an XY mixer Hamiltonian.
# Uses SparsePauliOp from qiskit. First generates a list of Pauli operators and then uses .from_sparse_list to generate the sparse Pauli operator.

def XY_mixer(num_qubits):

    # Constructing the XY mixer Hamiltonian.
    Paulis_mix = []
    for n in range(num_qubits-1):
        Paulis_mix.append(("XX", [n,n+1], 1/2))
        Paulis_mix.append(("YY", [n,n+1], 1/2))
    
    H_mix = SparsePauliOp.from_sparse_list(Paulis_mix, num_qubits)

    return H_mix

In [ ]:

# Constructs and returns an X mixer Hamiltonian.
# Uses SparsePauliOp from qiskit. First generates a list of Pauli operators and then uses .from_sparse_list to generate the sparse Pauli operator.

def X_mixer(num_qubits):

    # Constructing the X mixer Hamiltonian
    Paulis_mix = []
    for i in range(num_qubits):
        Paulis_mix.append(("X", [i], 1))
    
    H_mix = SparsePauliOp.from_sparse_list(Paulis_mix, num_qubits)
    
    return H_mix

In [ ]:

# Function that takes the commutator of two matrices X and Y. Will also work if X and Y are SparsePauliOps.

def commutator(X,Y):
    value = X@Y-Y@X
    return value

In [ ]:

# Function that takes a list of Pauli strings and restricts it to up to k-body terms.

def restrict_to_k_body(Pauli_list, k):

    A = []

    for p in Pauli_list:
        
        weight = sum(1 for char in p if char != "I")

        if weight <= k:
            
            A.append(p)

    return A

In [ ]:

# Function that calculates the operator pool A up a certain order and keep up to k-body terms for a given problem and mixer Hamiltonians.
# It assumes an adiabatic Hamiltonian.

def A_NC(H_prob, H_mix, order, k):

    # Calculates numerically so have to set the lambda function to a value. 
    # This can possiblily lead to unintented cancellations so change this value (not to 1) to check. 
    lambda_fn = 0.1
    # Define H_0 as an adiabatic Hamiltonian.
    H_0 = (1-lambda_fn)*H_mix + lambda_fn*H_prob
    # Derivative with respect to lambda for H_0.
    H_0_deriv = -H_mix + H_prob

    # Stores the Pauli strings of nested commutater in the sum.
    A_terms = []
    O = H_0_deriv

    # Iteratively calculates the nested commutaters in the sum up the 2*order-1 term.
    for i in range(2*order-1):

        O_i = commutator(H_0, O).simplify(atol=1e-10)
        O = O_i
        A_terms.append(O_i.paulis)

    # Stores the operator terms in the pool
    A = []
    for j in range(1, order+1):

        # Appends the odd terms in the sum to A.
        A.append(A_terms[2*j-2])

    # Flattens the list of lists into a list.
    A = list(itertools.chain.from_iterable(A))
    # Converts the Pauli objects to strings.
    A = [(p.to_label()) for p in A]
    # Gets rid of the Identity operator in the Pauli strings (e.g. 'XIIY becomes 'XY') as we are just interested in the actual operator terms 
    # that get summed over
    A = [p.replace('I', '') for p in A]
    # Gets rid of any duplicate Pauli strings in the list.
    A = list(dict.fromkeys(A))
    # Restricts the Pauli strings to k-body terms.
    A = restrict_to_k_body(A, k)

    # Returns A in alphabetically and with the shortest strings appearing first.
    return sorted(A, key=lambda p: (len(p), p))

In [ ]:

# Sets up the problem and mixer Hamiltonians and calls the A_NC function to get the operator pool A.

# Sets the number of qubits
num_qubits = 4

# Model parameters for Schwinger model.
a = 5/6
g = 1
m = 0.35
theta = 0
J = g**2*a/2
w = 1/(2*a)

# Constructing the Schwinger model Hamiltonian.
H_prob_Schwinger = Schwinger_Hamiltonian(num_qubits, J, w, m, theta)

# Constructing XY and X mixers
H_mix_XY = XY_mixer(num_qubits)
H_mix_X = X_mixer(num_qubits)

# Order of the expansion to go up to.
order = 2
# The number of k-body terms to go up to in the operator pool.
k = 3
# Calls A_NC function to get A.
A = A_NC(H_prob_Schwinger, H_mix_XY, order, k)
print(A)